# Defender feature engineering

This is the dedicated DEF research branch of `fantasy_premier_league.ipynb`. The root owns shared data construction and benchmark definitions; this notebook owns defender-specific features, diagnostics, model comparisons, and prospective challengers.

The extracted benchmark represents both scoring routes: attacking goal/assist potential and team-based clean-sheet or defensive-contribution potential. Results here do not change the root model automatically.

In [1]:
import pandas as pd
from IPython.display import display

from position_research import compare_models

POSITION = "DEF"
K = 10
PANEL_PATH = "fpl_feature_panel.pkl"

panel = pd.read_pickle(PANEL_PATH)
position_panel = panel.loc[panel["position"].eq(POSITION)].copy()
print(POSITION, position_panel.shape, "K =", K)
display(position_panel.groupby("season")["GW"].agg(["min", "max", "nunique"]))

DEF (20987, 95) K = 10


,min,max,nunique
season,,,
2024-25,1,38,38
2025-26,1,38,38
2026-27,1,11,11


## Extracted DEF feature set

This is the current root challenger moved into its own research branch. The next defender experiments belong here: separate clean-sheet and attacking routes, stronger-team priors, past goals conceded, defensive-contribution opportunity, and opponent set-piece weakness.

In [2]:
defender_features = [
    "minutes_ewm", "zero_min_rate_5", "start_rate_5", "minutes_60_rate",
    "minutes_form", "xg_per90_5", "xa_per90_5", "xg_per90_10",
    "xa_per90_10", "threat_per90_5", "creativity_per90_5",
    "def_contribution_rate_5", "bps_per90_5", "bonus_rate_5",
    "clean_sheet_rate_5", "team_xga_venue_5", "opp_xg_venue_5",
    "venue_defense_matchup", "team_clean_sheet_venue_rate_5",
    "n_fixtures", "n_home", "value",
]

missing = sorted(set(defender_features) - set(position_panel.columns))
assert not missing, f"Missing root features: {missing}"
print("DEF features:", len(defender_features))
display(pd.Series(defender_features, name="feature").to_frame())

DEF features: 22


,feature
0,minutes_ewm
1,zero_min_rate_5
2,start_rate_5
3,minutes_60_rate
4,minutes_form
5,xg_per90_5
6,xa_per90_5
7,xg_per90_10
8,xa_per90_10
9,threat_per90_5


## Leak-safe walk-forward benchmark

Each evaluated gameweek trains only on earlier rows. The target and predicted shortlist are tie-aware top K, and each training gameweek receives equal total weight. GW1-19 is development; GW20-38 is a historical check.

In [3]:
defender_summary, defender_detail = compare_models(
    position_panel,
    defender_features,
    K,
)
display(defender_summary)
display(
    defender_summary.pivot(index="model", columns="block", values="mean_hit_rate")
)

,block,model,mean_hit_rate,correct,predictions,weeks
1,development_gw1_19,logistic_regression,0.168421,32,190,19
0,development_gw1_19,gradient_boosting,0.136842,26,190,19
2,development_gw1_19,minutes_sort,0.116792,25,206,19
4,historical_check_gw20_38,logistic_regression,0.247368,47,190,19
3,historical_check_gw20_38,gradient_boosting,0.231579,44,190,19
5,historical_check_gw20_38,minutes_sort,0.215789,41,190,19


block,development_gw1_19,historical_check_gw20_38
model,,
gradient_boosting,0.136842,0.231579
logistic_regression,0.168421,0.247368
minutes_sort,0.116792,0.215789
